In [ ]:
import random
import sys
import numpy as np
import torch

from pathlib import Path
from pycocotools.coco import COCO
from torch.utils.data import DataLoader

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/ucu-ai-hw-2")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.data_prep import load_annotations
from scripts.data_cv import stratified_group_folds
from scripts.dataset_helpers import DetectionDataset, collate_data
from scripts.model_builders import build_frcnn_model
from scripts.experiment_runner import run_torch_cv
from scripts.experiment_results import model_training_results

DIRECTORY = PROJECT_ROOT / "data"
INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"

Mounted at /content/drive


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
FOLDS = 3
EPOCHS = 5
FRAME_STEP = 15
BATCH_SIZE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
train_frames, train_boxes = load_annotations(COCO(str(INFO_TRAIN)), split="train")
frames = stratified_group_folds(train_frames, train_boxes, n_splits=FOLDS, seed=SEED)
annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
empty_annotations = train_boxes.iloc[:0].copy()

loading annotations into memory...
Done (t=15.08s)
creating index...
index created!


In [ ]:
cv_fold_loaders = {}

kwargs_dataset = {
    "directory": DIRECTORY,
    "annotations_by_image": annotations_by_image,
    "empty_annotations": empty_annotations,
}

for fold in range(FOLDS):
    training_frames = (frames.loc[frames[f"train_fold_{fold}"].eq(True)].sort_values(["video", "frame_id"])
        .groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

    validation_frames = frames.loc[frames["validation_fold"].eq(fold)].reset_index(drop=True)

    train_dataset = DetectionDataset(frames=training_frames, validation=False, **kwargs_dataset)
    valid_dataset = DetectionDataset(frames=validation_frames, validation=True, **kwargs_dataset)

    cv_fold_loaders[fold] = {
        "train": DataLoader(train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            collate_fn=collate_data
        ),
        "valid": DataLoader(
            valid_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            collate_fn=collate_data
        )
    }

    print(f"Fold {fold}: {len(train_dataset)} training frames, {len(valid_dataset)} validation frames")

Fold 0: 1107 training frames, 7943 validation frames
Fold 1: 1094 training frames, 8088 validation frames
Fold 2: 1091 training frames, 8170 validation frames


In [ ]:
cv_kwargs = {
    "cv_fold_loaders": cv_fold_loaders,
    "annotations_path": INFO_TRAIN,
    "device": device,
    "seed": SEED,
    "epochs": EPOCHS,
    "lr": 1e-4,
    "weight_decay": 1e-4,
}

## baseline model

In [ ]:
baseline_histories, baseline_results = run_torch_cv(**cv_kwargs, model_method=build_frcnn_model,
    output_dir=PROJECT_ROOT / "outputs" / "frcnn_baseline", lr_scheduler=False)

In [ ]:
baselines_results_wrap = model_training_results(ROOT=PROJECT_ROOT, filename="frcnn_baseline", cv_folds=range(FOLDS))


Fold 0


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,1.257520,0.0001,0.084901,0.200863,0.031676
1,2,1.063862,0.0001,0.135092,0.295173,0.042535
2,3,0.994036,0.0001,0.130670,0.278883,0.043652
3,4,0.941511,0.0001,0.138030,0.294165,0.039764
4,5,0.898043,0.0001,0.147468,0.308827,0.052301



Fold 1


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,1.307781,0.0001,0.112596,0.249788,0.046229
1,2,1.099321,0.0001,0.131570,0.286099,0.051572
2,3,1.009177,0.0001,0.136085,0.293861,0.041439
3,4,0.951391,0.0001,0.166663,0.348690,0.055526
4,5,0.903169,0.0001,0.170524,0.351398,0.056580



Fold 2


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,1.244907,0.0001,0.088471,0.194347,0.042204
1,2,1.043111,0.0001,0.103797,0.225664,0.047077
2,3,0.960359,0.0001,0.105398,0.227095,0.039046
3,4,0.909814,0.0001,0.112619,0.239960,0.046442
4,5,0.873291,0.0001,0.108224,0.234563,0.044945



Best results per fold


,fold,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,0,5,0.898043,0.0001,0.147468,0.308827,0.052301
1,1,5,0.903169,0.0001,0.170524,0.351398,0.056580
2,2,4,0.909814,0.0001,0.112619,0.239960,0.046442



Metrics mean and std across folds


,mAP,AP50,AP_small
mean,0.143537,0.300062,0.051774
std,0.029152,0.056234,0.005090


## learning rate scheduler

In [ ]:
scheduler_histories, scheduler_results = run_torch_cv(**cv_kwargs, model_method=build_frcnn_model,
    output_dir=PROJECT_ROOT / "outputs" / "frcnn_baseline_scheduler", lr_scheduler=True, step_size=4)

loading annotations into memory...
Done (t=5.33s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.06s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=238.91s).
Accumulating evaluation results...
DONE (t=11.67s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.095
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.226
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.062
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.031
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.086
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.195
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.062
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.153
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxD

In [ ]:
scheduler_results_wrap = model_training_results(ROOT=PROJECT_ROOT, filename="frcnn_baseline_scheduler", cv_folds=range(FOLDS))


Fold 0


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,1.262799,0.00010,0.094612,0.226446,0.030930
1,2,1.058773,0.00010,0.121022,0.257580,0.037434
2,3,0.990670,0.00010,0.130103,0.290326,0.041555
3,4,0.935259,0.00010,0.138325,0.301541,0.036976
4,5,0.826102,0.00001,0.173233,0.358332,0.050640



Fold 1


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,1.303611,0.00010,0.105642,0.242363,0.042517
1,2,1.103986,0.00010,0.139040,0.300019,0.049109
2,3,1.012085,0.00010,0.130780,0.279377,0.043457
3,4,0.949037,0.00010,0.158974,0.338993,0.052850
4,5,0.820961,0.00001,0.184243,0.370209,0.058439



Fold 2


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,1.239027,0.00010,0.090363,0.199477,0.042184
1,2,1.043507,0.00010,0.107321,0.229627,0.047986
2,3,0.962561,0.00010,0.108249,0.234890,0.041098
3,4,0.903412,0.00010,0.109524,0.235854,0.045565
4,5,0.799078,0.00001,0.130530,0.267310,0.052389



Best results per fold


,fold,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,0,5,0.826102,0.00001,0.173233,0.358332,0.050640
1,1,5,0.820961,0.00001,0.184243,0.370209,0.058439
2,2,5,0.799078,0.00001,0.130530,0.267310,0.052389



Metrics mean and std across folds


,mAP,AP50,AP_small
mean,0.162669,0.331950,0.053823
std,0.028372,0.056294,0.004093
